# Regresion Lineal IV. Regularización. 

En el ejemplo anterior hemos visto la importancia de elegir correctamente el orden polinómico. Pero eso es algo que no podemos saber a priori.
Hubiese bastado con que el modelo no de tanta importancia a minimizar el error en los puntos de entrenamiento ya que eso lleva a overfitting y que penalizase aumentar el orden polinómico.
Es decir que aunque permitamos aproximaciones de la forma:
$$
h_\Theta(\boldsymbol{x}) = \theta_0  + \theta_1 x_1 + \theta_2 x_2 + \theta_3 x_1^2 + \theta_4 x_2^2 + \theta_5 x_1x_2 + \dots 
$$
"premiemos" que $\theta_i \longrightarrow 0$.

Este comportamiento no es exclusivo de la regresión polinómica. Supongamos que tenemos un *dataset* con un gran número de atributos/características de entrada:

* $x_1 \rightarrow$ Tamaño en $\text{m}^2$
* $x_2 \rightarrow$ Número de habitaciones.
* $x_3 \rightarrow$ Número de baños.
* $x_4 \rightarrow$ Distancia al metro más próximo.
* ...
* $x_{100} \rightarrow$ Número de enchufes en la cocina.

De modo que, en el caso de hacer aproximación lineal, tendríamos: $h_\Theta(\boldsymbol{X}) = \displaystyle\sum_{i=0}^{100} \theta_i x_i$ 

Seguramente hay variables que no aportan gran cosa o características que se pueden combinar (distancia a hospital + distancia escuela ¿código postal?)

La idea detrás de la regularización es premiar que $\theta_i \ll 1$ 

### Ecuaciones de la Regularización. 

Vamos a ver como incluir la regularización en la función de coste de la aproximación. Vamos a usar un tipo particular de regularización, que es la regularización de **Ridge**, para ilustrar el proceso. Luego discutiremos otras maneras de regularizar.


Empezamos modificando la función de coste para incluir la penalización por tener pesos grandes.
La expresión original
$$
J(\Theta) = \dfrac{1}{m} \sum_{j=1}^m (h_\Theta(\boldsymbol{x^j})-y^j)^2
$$
solo premia minimizar el error en los puntos de entrenamiento, así que la modificamos.
Definimos la nueva función de coste:
$$
J(\Theta) = \dfrac{1}{m} \left( \displaystyle\sum_{j=1}^m (h_\Theta(\boldsymbol{x^j})-y^j)^2 + \lambda \displaystyle\sum_{i=0}^{n} \theta_i^2 \right)
$$

Es importante tener en cuenta que: 
* En la penalización se excluye $\theta_0$. Este término no influye en la complejidad del modelo, solo controla el desplazamiento o ajuste vertical del modelo.
* El valor de $\lambda$ determina cuánto impacto tiene esta penalización en el proceso de ajuste del modelo:
  * Si $\lambda$ es muy grande, el modelo prioriza reducir el tamaño de los coeficientes, lo que puede llevar a un modelo subajustado (demasiado simple).
  * Si $\lambda$ es muy pequeño o igual a cero, la regularización no tiene efecto, y el modelo puede sobreajustarse (es decir, adaptarse en exceso a los datos de entrenamiento).
  * Para determinar su valor óptimo: *Train*-**Validation**-*Test*
* Es imprescindible **normalizar** las variables antes de regularizar. La regularización penaliza los coeficientes del modelo. Si las variables no están normalizadas, las que tienen escalas más grandes pueden dominar el ajuste del modelo y, por ende, influir desproporcionadamente en la penalización. Normalizando las variables, el impacto de $\lambda$ se vuelve más consistente y comparable entre diferentes modelos.

### Implementación de la Regularización.

#### Ecuaciones Normales.

Minimicemos la nueva expresión de la función de error:
$$
J(\Theta) = \dfrac{1}{m} \displaystyle\sum_{j=1}^m (h_\Theta(\boldsymbol{x^j})-y^j)^2 + \dfrac{\lambda}{m} \displaystyle\sum_{i=0}^{n} \theta_i^2 
$$
De nuevo buscamos $\nabla_\Theta J(\Theta) = 0$

Para $i\ne 0$ tenemos:

$$
\dfrac{\partial}{\partial \theta_i}J(\Theta) = \dfrac{2}{m} \displaystyle\sum_{j=1}^m (h_\Theta(\boldsymbol{x^j})-y^j)x_i^j + \boxed{ \dfrac{2\lambda}{m} \theta_i} = 0
$$

La expresión es idéntica a la que ya teníamos salvo por el nuevo término que aparece recuadrado.
Si la escribimos en forma matricial obtenemos:
$$
\boxed{
\left(X^TX +\lambda D\right)\Theta = X^T Y } \text{   con }
$$

$$
D = \begin{pmatrix}
0 & 0 & 0 & \dots & 0\\
0 & 1 & 0 & \dots & 0\\
0 & 0 & 1 &  \dots & 0\\
\vdots & \vdots &  & \dots & \vdots\\
0 & 0 & 0 & \dots & 1
\end{pmatrix} 
\hspace{0.5cm}
D \in \mathbb{M}^{(n+1)\times(n+1)}(\mathbb{R})
$$

Una ventaja adicional de la regularización es que hace a la matriz del sistema a resolver más diagonal dominante con las implicaciones numéricas que tiene respecto al cálculo de la inversa.


#### Descenso de gradiente.

Vamos a partir de la ecuación vectorial.

$$
\Theta^{k+1} = \Theta^k  - \alpha \nabla_{\Theta}J(\Theta^k)
$$

que matricialmente se puede escribir componente a componente como:
$$
\begin{pmatrix}
\displaystyle\theta_0^{k+1} \\
\\
\displaystyle\theta_1^{k+1} \\ 
\\
\vdots \\ 
\\
\displaystyle\theta_n^{k+1}
\end{pmatrix} =
\begin{pmatrix}
\displaystyle\theta_0^k \\ 
\\
\displaystyle\theta_1^K \\ 
\\
\vdots \\ 
\\
\displaystyle\theta_n^k
\end{pmatrix}
-
\alpha 
\begin{pmatrix}
\displaystyle\dfrac{\partial J (\Theta^k)}{\partial \theta_0} \\ 
\displaystyle\dfrac{\partial J (\Theta^k)}{\partial \theta_1} \\ 
\vdots \\ 
\displaystyle\dfrac{\partial J (\Theta^k)}{\partial \theta_n}
\end{pmatrix}
$$

* Para $i = 0$:
$$
\theta_0^{k+1} = \theta_0^{k} -\alpha \dfrac{2}{m} \sum_{j=1}^m (h_{\Theta^k}(\boldsymbol{x}^j)-y^j)  \hspace{0.5cm} \text{recuerda que: }x_0^j = 1 
$$
* Para $i \ne 0$:
$$
\theta_i^{k+1} = \theta_i^{k} -\alpha \left( \dfrac{2}{m} \sum_{j=1}^m (h_{\Theta^k}(\boldsymbol{x}^j)-y^j)x_i^j + \lambda\theta_i^k \right)
$$
Sacando factor común:
$$
\theta_i^{k+1} = (1-\frac{2\alpha\lambda}{m})\theta_i^{k}  -\alpha \left( \dfrac{2}{m} \sum_{j=1}^m (h_{\Theta^k}(\boldsymbol{x}^j)-y^j)x_i^j \right)
$$
siendo $(1-\frac{2\alpha\lambda}{m})$ un factor menor que la unidad.

Si recuperamos la forma vectorial:
$$
\boxed{\Theta^{k+1} = D \Theta^k  - \alpha\left( \boldsymbol{X}^T\boldsymbol{X}\Theta  - \boldsymbol{X}^TY\right)} \text{   con }
$$

$$
D = \begin{pmatrix}
1\\
1-\frac{2\alpha\lambda}{m}\\
1-\frac{2\alpha\lambda}{m}\\
\vdots \\
1-\frac{2\alpha\lambda}{m}
\end{pmatrix} 
$$

### Tipos de Regularización:

Aunque hay más (entre otros *Drop out*) nos vamos a fijar solamente en los tres tipos de regularización más empleados.

* **Ridge** (también llamada regularización **L2**) añade una penalización al término de la función de costo, basada en la suma de los cuadrados de los coeficientes del modelo:

    * Mantiene todos los coeficientes: No tiende a llevar coeficientes exactamente a cero, sino que los reduce proporcionalmente, lo que significa que todas las variables permanecen en el modelo. Eso puede ser una ventaja o un inconveniente ya que puede ser más difícil interpretar el resultado cuando hay muchas características irrelevantes.
    * Se usa en casos en los que hay muchas características que pueden estar correlacionadas entre sí (multicolinealidad), pero no queremos descartar ninguna variable completamente.
$$
J(\Theta) = \dfrac{1}{m} \displaystyle\sum_{j=1}^m (h_\Theta(\boldsymbol{x^j})-y^j)^2 + \dfrac{\lambda}{m} \displaystyle\sum_{i=0}^{n} \theta_i^2 
$$

* **Lasso** (Least Absolute Shrinkage and Selection Operator) añade una penalización basada en la suma de los valores absolutos de los coeficientes:

    * Lasso tiende a forzar algunos coeficientes a cero, lo que elimina efectivamente las variables menos importantes del modelo. Esto introduce sparsidad, haciendo que el modelo sea más interpretable y eficiente.
    * Puede introducir un sesgo mayor que Ridge, especialmente cuando las características son muy numerosas o correlacionadas.
    * El hecho de que Lasso involucre valores absolutos en su penalización hace que la función de costo no sea derivable en ciertos puntos (donde los coeficientes son exactamente cero). En la práctica se usan algoritmos especificos como: *Coordinate Descent*, *Least Angle Regression* o *Proximal Gradient*.
$$
J(\Theta) = \dfrac{1}{m} \displaystyle\sum_{j=1}^m (h_\Theta(\boldsymbol{x^j})-y^j)^2 + \dfrac{\lambda}{m} \displaystyle\sum_{i=0}^{n} |\theta_i| 
$$



* **Elastic-Net** Elastic Net combina las penalizaciones de L1 (Lasso) y L2 (Ridge), de modo que se benefician de las ventajas de ambas:

    * Combina las mejores cualidades de Lasso y Ridge: Proporciona sparsidad al mismo tiempo que maneja multicolinealidad.
    * Más complejidad en la selección de parámetros: Hay dos parámetros de regularización que ajustar

$$
J(\Theta) = \dfrac{1}{m} \displaystyle\sum_{j=1}^m (h_\Theta(\boldsymbol{x^j})-y^j)^2 +  \dfrac{\lambda_1}{m} \displaystyle\sum_{i=0}^{n} \theta_i^2 + \dfrac{\lambda_2}{m} \displaystyle\sum_{i=0}^{n} |\theta_i| 
$$

#### Importamos librerías

In [1]:
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import random

#### Implementación Ecuación Normal.

De momento solo vamos a implementar la ecuación normal modificada para el caso de **Ridge** 

Implementaremos:

* `normal_equations_ridge(X, y, alpha = 1.)`.



In [ ]:
def normal_equations_ridge(X, y, alpha = 1.):
    """
    X: el conjunto de datos de entrada en la matriz X(m,n)
        m: número de datos. 
        n: número de variables predictoras o atributos.
    y: Vector con las imagenes de los datos de entrada conocidas. 
    alpha: coeficiente de regularización

    Resuelvo la ecuación normal [(X^T)X + alpha*D] Theta = (X^T)Y
    Return Theta: vector de coeficientes de la regresión lineal
    """
    
    return None


Seguimos con la  función:

* `polinomio()`: Una vez conocidos los coeficientes, devuelve la predicción polinómica dado un conjunto de datos de entrada.


In [ ]:

def polinomio(coef, x):
    

    
    return value




Vamos a ver un ejemplo de **Regresión Polinómica** usando la regularización de Ridge que es la que hemos implementado manualmente:

In [ ]:
# generamos un conjunto de datos en el intervalo [-2,2]
a = -2.
b = 2.
n_points = 30
x_i = np.random.rand(n_points)*(b-a) + a

# generamos sus imagenes a partir de la función y(x) = 3*x^5 -2*x + ruido 
# El ruido lo genero normalmente distribuido entre 
y_x = 3*x_i**5 -2*x_i
max_val = max(abs(y_x))

ruido = np.random.normal(-(max_val)/10, max_val/10, n_points)
y_i = y_x + ruido

# Divido en training y test
n_train = 16
order = list(range(len(x_i)))
np.random.shuffle(order)

X_train = np.array([x_i[i] for i in order[0:n_train]])
y_train = np.array([y_i[i] for i in order[0:n_train]])

X_test = np.array([x_i[i] for i in order[n_train:]])
y_test = np.array([y_i[i] for i in order[n_train:]])


# Representamos los puntos iniciales
sns.scatterplot(x=x_i,y=y_i,  marker = 'o' , edgecolor='blue', facecolor='none', label= 'Puntos originales', s= 40 )
sns.scatterplot(x=X_train,y=y_train,  marker = 'o' , color = 'red', label= 'Puntos entrenamiento', s= 20 )
sns.scatterplot(x=X_test,y=y_test,  marker = 'o' , color = 'black', label= 'Puntos test', s= 20 )

plt.title('Puntos ')
plt.xlabel('x')
plt.ylabel('y')
plt.xlim(a, b)


#### Calculamos los coeficientes de la regresión polinómica con y sin regularización.

Para ello jugamos con el parametro $\lambda$ (`alpha` en el código) 
* $\lambda = 0$ implica no regularizar.

Fijamos un orden polinómico de 12. 

In [ ]:
orden_polinomico = 12
# Coeficientes sin regularizar
alpha = 0
# ********************
# Theta_no_reg = 
# ********************

alpha = 10
# ********************
# Theta_no_reg = 
# ********************



####  Representación Gráfica

Representamos en una misma grafica:
* Puntos originales.
* Puntos de *test*
* Polinomio de grado 12 sin regularización.
* Polinomio de grado 12 con regularización por Ridge.

In [ ]:

x_value = np.linspace(a,b,300)
y_value = 
y_value_reg = )

# Representamos los puntos iniciales
plt.figure(figsize=(12, 6))
sns.lineplot(x=x_value,y=y_value_reg, color = 'blue', label= 'Polinomio aproximado grado ' +str(orden_polinomico)+ ' usando Ridge (lambda = 10).' )
sns.lineplot(x=x_value,y=y_value, color = 'green', label= 'Polinomio aproximado grado ' +str(orden_polinomico)+ ' sin regularizar' )
sns.scatterplot(x=X_train,y=y_train,  marker = 'o' , color = 'red', label= 'Puntos entrenamiento', s= 20 )
sns.scatterplot(x=x_i,y=y_i,  marker = 'o' , edgecolor='blue', facecolor='none', label= 'Puntos originales', s= 40 )
plt.xlabel('x')
plt.ylabel('y')
plt.xlim(a-0.2,b+0.2)
plt.ylim(min(y_i)-5, max(y_i)+5)
plt.title('Regularización')
